In [ ]:
NAMA  : AHMAD DZAKY HERYADI
KELAS : 06TPLE006
NIM   : 231011400278
UTS   : TEKNIK KOMPILASI

In [ ]:
1.
class AST:
    pass

class BinOp(AST):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class Num(AST):
    def __init__(self, value):
        self.value = value

class Var(AST):
    def __init__(self, name):
        self.name = name

In [ ]:
2
import re

class MiniCompiler:
    def __init__(self, source, env):
        # TUGAS 1: Perbarui regex di bawah ini agar mengenali simbol '^'
        self._tokens = iter(re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()\-^]', source) + ['?'])
        self._current = None
        self._env = env
        self._temp_count = 0
        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        if self._current != expected and not (expected == "ID" and self._current.isalnum()):
            raise ParserError(f"Expected {expected}, found {self._current}")
        token = self._current
        self.advance()
        return token

    def factor(self):
        token = self._current
        if token is not None and token.replace('.', '', 1).isdigit():
            self.advance()
            return Num(float(token) if '.' in token else int(token))
        elif token and token.isalpha():
            if token not in self._env:
                raise ParserError(f"Semantic Error: Undefined variable '{token}'")
            self.advance()
            return Var(token)
        elif token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node
        raise ParserError(f"Unexpected token: {token}")

    # TUGAS 2: Implementasikan fungsi power()
    # Gunakan pola yang sama dengan term(), namun untuk operator '^'
    def power(self):
        node = self.factor()
        # Tulis logika perulangan while untuk '^' di sini

        while self._current == '^':
        op = self._current
        self.eat('^')
        node = BinOp(node, op, self.factor())

        return node

    def term(self):
        # TUGAS 3: Hubungkan hierarki ke self.power()
        node = self.power() # Ubah ini
        while self._current in ('*', '/'):
            op = self._current
            self.advance()
            node = BinOp(node, op, self.power()) # Dan sesuaikan pemanggilan kanannya
        return node

    def expr(self):
        node = self.term()
        while self._current in ('+', '-'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.term())
        return node

    def generate_tac(self, node):
        if isinstance(node, Num): return str(node.value)
        if isinstance(node, Var): return node.name

        left_val = self.generate_tac(node.left)
        right_val = self.generate_tac(node.right)

        self._temp_count += 1
        temp_name = f"t{self._temp_count}"
        print(f"{temp_name} = {left_val} {node.op} {right_val}")
        return temp_name

In [15]:
import re

class AST:
    pass


class BinOp(AST):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right


class Num(AST):
    def __init__(self, value):
        self.value = value


class Var(AST):
    def __init__(self, name):
        self.name = name


class ParserError(Exception):
    pass

class MiniCompiler:

    def __init__(self, source, env):

        # tokenizer
        self._tokens = iter(
            re.findall(
                r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()\-^]',
                source
            ) + ['?']
        )

        self._current = None
        self._env = env
        self._temp_count = 0

        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def eat(self, token):

        if self._current == token:
            self.advance()

        else:
            raise ParserError(
                f"Expected '{token}', got '{self._current}'"
            )

    # factor -> NUMBER | IDENTIFIER | '(' expr ')'
    def factor(self):

        token = self._current

        # angka
        if token.isdigit():

            self.advance()
            return Num(int(token))

        # variabel
        elif re.match(r'[a-zA-Z_]\w*', token):

            if token not in self._env:
                raise Exception(
                    f"Semantic Error: variabel '{token}' tidak ditemukan"
                )

            self.advance()
            return Var(token)

        # kurung
        elif token == '(':

            self.eat('(')

            node = self.expr()

            self.eat(')')

            return node

        raise ParserError(f"Unexpected token: {token}")

    # power -> factor (^ factor)*
    def power(self):

        node = self.factor()

        while self._current == '^':

            op = self._current

            self.eat('^')

            node = BinOp(
                node,
                op,
                self.factor()
            )

        return node

    # term -> power ((*|/) power)*
    def term(self):

        node = self.power()

        while self._current in ('*', '/'):

            op = self._current

            self.advance()

            node = BinOp(
                node,
                op,
                self.power()
            )

        return node

    # expr -> term ((+|-) term)*
    def expr(self):

        node = self.term()

        while self._current in ('+', '-'):

            op = self._current

            self.advance()

            node = BinOp(
                node,
                op,
                self.term()
            )

        return node

    def new_temp(self):

        self._temp_count += 1

        return f"t{self._temp_count}"

    def generate_tac(self, node):

        # number
        if isinstance(node, Num):
            return str(node.value)

        # variable
        elif isinstance(node, Var):
            return node.name

        # binary operation
        elif isinstance(node, BinOp):

            left = self.generate_tac(node.left)

            right = self.generate_tac(node.right)

            temp = self.new_temp()

            print(f"{temp} = {left} {node.op} {right}")

            return temp

In [16]:
source_code = "a ^ 2 + b * c"
symbol_table = {'a': 5, 'b': 10, 'c': 2}

try:
    print(f"Input: {source_code}")

    compiler = MiniCompiler(source_code, symbol_table)
    ast_root = compiler.expr()

    print("\n--- Output Three Address Code (TAC) ---")
    compiler.generate_tac(ast_root)

except Exception as e:
    print(f"Error: {e}")

Input: a ^ 2 + b * c

--- Output Three Address Code (TAC) ---
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2


In [ ]:
4.
a. Mengapa power() dipanggil di dalam term()?
- Karena operator ^ memiliki prioritas (precedence) lebih tinggi dibanding * dan /.
- Jika power() tidak dipanggil di dalam term(), parser bisa salah membaca ekspresi.

b. Apa yang terjadi jika variabel z tidak ada di symbol_table?
- Pada fase Analisis Semantik, compiler akan menghasilkan error karena variabel tersebut belum dideklarasikan atau tidak dikenal.
Contoh:
source_code = "z + 1"
Jika z tidak ada dalam: symbol_table
Maka akan muncul: Semantic Error: variabel 'z' tidak ditemukan

c. Mengapa instruksi a ^ 2 harus muncul sebelum + dalam TAC?
- Karena TAC mengikuti urutan evaluasi berdasarkan struktur AST dan prioritas operator.
- Pada ekspresi:a ^ 2 + b * c Compiler harus menghitung sub-ekspresi lebih dahulu:
1. a ^ 2
2. b * c
3. hasil keduanya dijumlahkan
Sehingga TAC menjadi:
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2
Jika operasi + dilakukan lebih dulu, maka operand belum tersedia dan hasil evaluasi menjadi salah.